# 09.07 - Small CNN Coding

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** a small CNN baseline trained on a synthetic image classification dataset, with train and validation loss recorded.

Today turns Day 08 shape math into a real model: conv blocks, batch norm, activation, pooling, classifier head, `CrossEntropyLoss`, and a complete train/eval loop.


## Core Ideas

A compact CNN classifier usually has:

- a feature extractor: repeated convolution blocks
- nonlinear activations: usually ReLU or GELU
- normalization: often BatchNorm for small CNNs
- pooling or stride: shrinks spatial size
- classifier head: flatten or global average pool, then linear layers

For multi-class classification with class IDs, use `nn.CrossEntropyLoss`. The model should output raw logits shaped `[batch, num_classes]`; do not apply softmax before the loss.


In [1]:
import random
import numpy as np

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset, random_split
    TORCH_AVAILABLE = True
except ImportError:
    torch = None
    nn = None
    DataLoader = None
    TensorDataset = None
    TORCH_AVAILABLE = False
    print("PyTorch is not installed. Complete this notebook in an environment with torch.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)
else:
    device = None


device: cpu


## Prepared Image Data

Run this cell before the exercises. The synthetic image data is provided so the learning work starts at dataset inspection, batching, modeling, and training.


In [2]:
def make_synthetic_image_dataset(n_per_class=80, image_size=16, noise=0.12, seed=42):
    if not TORCH_AVAILABLE:
        raise ImportError("PyTorch is required for this exercise.")

    generator = torch.Generator().manual_seed(seed)
    images = []
    labels = []

    for class_id in range(3):
        for _ in range(n_per_class):
            img = torch.zeros(1, image_size, image_size, dtype=torch.float32)
            if class_id == 0:
                img[:, :, image_size // 2 - 1:image_size // 2 + 1] = 1.0
            elif class_id == 1:
                img[:, image_size // 2 - 1:image_size // 2 + 1, :] = 1.0
            else:
                for i in range(image_size):
                    img[:, i, i] = 1.0
                    if i + 1 < image_size:
                        img[:, i, i + 1] = 1.0

            img = img + noise * torch.randn(img.shape, generator=generator)
            img = img.clamp(0.0, 1.0)
            images.append(img)
            labels.append(class_id)

    X = torch.stack(images)
    y = torch.tensor(labels, dtype=torch.long)
    perm = torch.randperm(len(y), generator=generator)
    return X[perm], y[perm]

if TORCH_AVAILABLE:
    X, y = make_synthetic_image_dataset()
    print("X:", X.shape, X.dtype)
    print("y:", y.shape, y.dtype, y.unique().tolist())


X: torch.Size([240, 1, 16, 16]) torch.float32
y: torch.Size([240]) torch.int64 [0, 1, 2]


## Exercise 09-A: Dataset Sanity Checks

Use the prepared tensors `X` and `y`. Write a small summary helper that checks image shape, label shape, dtypes, and label values before the data reaches a model.


In [25]:
# TODO 09-A
# Implement summarize_dataset(X, y).
# Return a dictionary with image_shape, label_shape, image_dtype, label_dtype, and label_values.

def summarize_dataset(X, y):
    image_shape = tuple(X.shape)
    label_shape = tuple(y.shape)
    image_dtype = X.dtype
    label_dtype = y.dtype
    label_values = y.unique().tolist()
    return {
        "image_shape" : image_shape,
        "label_shape" : label_shape,
        "image_dtype" : image_dtype,
        "label_dtype" : label_dtype,
        "label_values" : label_values
    }

summarize = summarize_dataset(X,y)
print(summarize)
# TODO: summarize the prepared X and y tensors.


{'image_shape': (240, 1, 16, 16), 'label_shape': (240,), 'image_dtype': torch.float32, 'label_dtype': torch.int64, 'label_values': [0, 1, 2]}


## Exercise 09-B: DataLoaders

Split the synthetic data into train and validation sets. Use shuffled training batches and deterministic validation batches.


In [4]:
# TODO 09-B
# Implement build_loaders(X, y, batch_size=32, train_frac=0.8).
# Return train_loader and val_loader.

def build_loaders(X, y, batch_size=32, train_frac=0.8, seed=42):
    if not TORCH_AVAILABLE:
        raise ImportError("PyTorch is required for this exercise.")
    dataset = TensorDataset(X,y)

    train_size = int(len(dataset) * train_frac)
    val_size = len(dataset) - train_size

    generator = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(
        dataset,
        (train_size,val_size),
        generator = generator
    )
    train_loader = DataLoader(train_ds, batch_size = batch_size, shuffle = True)
    val_loader = DataLoader(val_ds, batch_size = batch_size, shuffle = False)
    return train_loader, val_loader

train_loader, val_loader = build_loaders(X,y)
# TODO: create train_loader and val_loader.


## Exercise 09-C: CNN Building Blocks

Implement a reusable conv block:

`Conv2d -> BatchNorm2d -> ReLU`

Then build a small CNN with two pooling stages and a classifier head.


In [13]:
# TODO 09-C
# Implement ConvBlock and SmallCNN.
# SmallCNN should output logits with shape [B, num_classes].

class ConvBlock(nn.Module if TORCH_AVAILABLE else object):
    def __init__(self, in_channels, out_channels):
        if not TORCH_AVAILABLE:
            return
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels = self.in_channels,
                out_channels = self.out_channels,
                kernel_size = 3,
                stride = 1,
                padding = 1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.block(x)

class SmallCNN(nn.Module if TORCH_AVAILABLE else object):
    def __init__(self, num_classes=3):
        if not TORCH_AVAILABLE:
            return
        super().__init__()
        self.num_classes = num_classes
        self.features = nn.Sequential(
            ConvBlock(1, 16),
            nn.MaxPool2d(kernel_size = 2, stride = 2),
            ConvBlock(16,32),
            nn.MaxPool2d(kernel_size = 2, stride = 2),
            ConvBlock(32,64),
        )
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Linear(in_features = 64, out_features = self.num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, start_dim = 1)
        logits = self.classifier(x)
        return logits

# TODO: instantiate the model and test one forward pass.
model = SmallCNN(num_classes = 10)
dummy_x = torch.randn(8,1,64,64)
logits = model(dummy_x)
print(logits)

tensor([[-0.5211, -0.1335, -0.1487,  0.4207,  0.2747, -0.0222, -0.3952,  0.4136,
          0.0636, -0.4328],
        [-0.4839, -0.1142, -0.1634,  0.4156,  0.2684, -0.0146, -0.4032,  0.4256,
          0.0898, -0.4627],
        [-0.4767, -0.0938, -0.1710,  0.4181,  0.2528,  0.0169, -0.3982,  0.3726,
          0.0591, -0.4532],
        [-0.4747, -0.1164, -0.1870,  0.4198,  0.2987, -0.0042, -0.3815,  0.3938,
          0.0789, -0.4723],
        [-0.4820, -0.1184, -0.1342,  0.4422,  0.2830, -0.0392, -0.3551,  0.4001,
          0.1061, -0.4537],
        [-0.5112, -0.0407, -0.1791,  0.4712,  0.2519, -0.0671, -0.3910,  0.3698,
          0.1135, -0.4673],
        [-0.4674, -0.1105, -0.1751,  0.4502,  0.2563, -0.0137, -0.3954,  0.3860,
          0.1038, -0.4566],
        [-0.5043, -0.1032, -0.1709,  0.4415,  0.2816, -0.0299, -0.3947,  0.4057,
          0.0667, -0.4247]], grad_fn=<AddmmBackward0>)


## Exercise 09-D: Train and Evaluate

Write one training epoch and one evaluation function. Track average loss and accuracy.


In [29]:
# TODO 09-D
# Implement train_one_epoch and evaluate.
# Use model.train() for training and model.eval() with torch.no_grad() for validation.

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for images, label in loader : 
        images = images.to(device)
        label = label.to(device)
        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits,label)
        loss.backward()

        optimizer.step()
        batch_size = label.size(0)
        total_loss += loss.item() * batch_size

        predictions = logits.argmax(dim = 1)
        correct = (predictions == label).sum().item()
        total_correct += correct
        total_samples += batch_size
    avg_loss = total_loss/total_samples
    accuracy = total_correct/total_samples
    return avg_loss, accuracy

def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0
    with torch.no_grad() : 
        for images, label in loader : 
            images = images.to(device)
            label = label.to(device)

            logits = model(images)
            loss = criterion(logits,label)

            batch_size = label.size(0)
            total_loss += loss.item() * batch_size

            predictions = logits.argmax(dim = 1)
            correct = (predictions == label).sum().item()
            total_correct += correct
            total_samples += batch_size
    avg_loss = total_loss/total_samples
    accuracy = total_correct/total_samples
    return {
        "loss" : avg_loss,
        "accuracy" : accuracy
    }

model = SmallCNN()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 1e-3
)
EPOCH = 10
metrics = []
for i in range(EPOCH) : 
    avg_loss, accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    metrics.append({
        "loss" : avg_loss,
        "accuracy" : accuracy
    })
print(metrics)
# TODO: train for a few epochs and append metrics to history.


[{'loss': 0.743417223294576, 'accuracy': 0.8697916666666666}, {'loss': 0.34524724384148914, 'accuracy': 1.0}, {'loss': 0.20796001454194388, 'accuracy': 1.0}, {'loss': 0.14231117318073908, 'accuracy': 1.0}, {'loss': 0.10836337755123775, 'accuracy': 1.0}, {'loss': 0.07779039318362872, 'accuracy': 1.0}, {'loss': 0.07030656064550082, 'accuracy': 1.0}, {'loss': 0.05680559823910395, 'accuracy': 1.0}, {'loss': 0.04576200184722742, 'accuracy': 1.0}, {'loss': 0.03934374389549097, 'accuracy': 1.0}]


## Exercise 09-E: Record Curves

Train the CNN for a small number of epochs and record `train_loss`, `train_acc`, `val_loss`, and `val_acc`.


In [19]:
# TODO 09-E
# Create model, criterion, optimizer.
# Train for 5 epochs and store dictionaries in history.
model = SmallCNN()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 1e-3
)
EPOCH = 10
history = []
for i in range(EPOCH) : 
    avg_loss, accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    history.append({
        "train_loss" : avg_loss,
        "train_accuracy" : accuracy,
        "val_loss" : val_loss,
        "val_accuracy" : val_acc
    })

print(history)


[{'train_loss': 0.7364185154438019, 'train_accuracy': 0.8333333333333334, 'val_loss': 1.0362945397694905, 'val_accuracy': 0.4583333333333333}, {'train_loss': 0.3030911659200986, 'train_accuracy': 1.0, 'val_loss': 0.9265944361686707, 'val_accuracy': 0.7291666666666666}, {'train_loss': 0.18224456906318665, 'train_accuracy': 1.0, 'val_loss': 0.73100346326828, 'val_accuracy': 1.0}, {'train_loss': 0.12903785208861032, 'train_accuracy': 1.0, 'val_loss': 0.4825161596139272, 'val_accuracy': 1.0}, {'train_loss': 0.09974729021390279, 'train_accuracy': 1.0, 'val_loss': 0.2726161579291026, 'val_accuracy': 1.0}, {'train_loss': 0.0742857785274585, 'train_accuracy': 1.0, 'val_loss': 0.14458332459131876, 'val_accuracy': 1.0}, {'train_loss': 0.058315541595220566, 'train_accuracy': 1.0, 'val_loss': 0.08072435607512791, 'val_accuracy': 1.0}, {'train_loss': 0.04918523505330086, 'train_accuracy': 1.0, 'val_loss': 0.04877114792664846, 'val_accuracy': 1.0}, {'train_loss': 0.047178504367669426, 'train_accurac

## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 09 tests passed`.


In [30]:
def run_day09_tests():
    if not TORCH_AVAILABLE:
        print("Day 09 tests skipped because PyTorch is not installed.")
        return

    required_names = [
        "make_synthetic_image_dataset",
        "summarize_dataset",
        "build_loaders",
        "ConvBlock",
        "SmallCNN",
        "train_one_epoch",
        "evaluate",
    ]
    for name in required_names:
        assert name in globals(), f"Missing function or class: {name}"
        assert callable(globals()[name]), f"{name} must be callable"

    X_test, y_test = make_synthetic_image_dataset(n_per_class=6, image_size=16, seed=123)
    assert X_test.shape == (18, 1, 16, 16), f"Unexpected X shape: {X_test.shape}"
    assert y_test.shape == (18,), f"Unexpected y shape: {y_test.shape}"
    assert X_test.dtype == torch.float32
    assert y_test.dtype == torch.long
    assert set(y_test.tolist()) == {0, 1, 2}

    summary = summarize_dataset(X_test, y_test)
    assert summary["image_shape"] == (18, 1, 16, 16)
    assert summary["label_shape"] == (18,)
    assert summary["image_dtype"] == torch.float32
    assert summary["label_dtype"] == torch.long
    assert summary["label_values"] == [0, 1, 2]

    train_loader_test, val_loader_test = build_loaders(X_test, y_test, batch_size=6, train_frac=0.67, seed=123)
    xb, yb = next(iter(train_loader_test))
    assert xb.ndim == 4 and xb.shape[1:] == (1, 16, 16)
    assert yb.dtype == torch.long

    model_test = SmallCNN(num_classes=3).to(device)
    with torch.no_grad():
        logits = model_test(xb.to(device))
    assert logits.shape == (xb.shape[0], 3), f"Unexpected logits shape: {logits.shape}"

    criterion_test = nn.CrossEntropyLoss()
    optimizer_test = torch.optim.Adam(model_test.parameters(), lr=0.01)
    train_loss, train_acc = train_one_epoch(model_test, train_loader_test, criterion_test, optimizer_test, device)
    assert isinstance(train_loss, float)
    assert 0.0 <= train_acc <= 1.0

    metrics = evaluate(model_test, val_loader_test, criterion_test, device)
    assert {"loss", "accuracy"}.issubset(metrics.keys())
    assert isinstance(metrics["loss"], float)
    assert 0.0 <= metrics["accuracy"] <= 1.0

    print("Day 09 tests passed")

run_day09_tests()


Day 09 tests passed


## Day 09 Checklist

Before trusting a CNN baseline, verify batch shape, logits shape, label dtype, loss decreases, validation is run with `eval()` and `no_grad()`, and the recorded metrics are validation metrics rather than training-only numbers.
